# Suite CHG — Changes: what a transaction changed, observed at its commit

A journal, a viewer or a layout engine needs to know exactly what a transaction changed, which its expression, a query that writes, does not say ([CAP-270: a transaction's changes observed at its commit](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-270)). `Changes.observe(store, observer)` gives `observer` the effect of each transaction that commits having changed something, and returns a function that ends it. The effect is written in the journal language: a transaction expression of the store's own schemas, one upsert, whose symbols are bound to the objects held before by their references and to the objects created by their schemas, and whose spec removes and inserts relation entries and sets and clears properties, net of what the transaction undid within itself. Applied to the store as it was, it makes the same changes; applied again, none.

Python and TypeScript run this suite with the same case IDs. The case setup is below.

In [ ]:
import { Builders as B, Changes, JSON as Json, Plain, Proxies } from "@mbse/schemas/Framework";
import { ValueError } from "@mbse/schemas/Framework/Errors";
import { assert, raises } from "./support.js";

/** crm.Person, owned by the singleton book by key, with a name and a home, who knows persons since a year, and pins
 * persons through no adjacency by name; crm.Note, counted, about a person. */
function crm(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("crm/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Home: S.OfObject, Knows: S.OfRelation, Person: S.OfObject, Holds: S.OfRelation, Book: S.OfObject,
      Note: S.OfObject, About: S.OfRelation, Pinned: S.OfRelation })
    .spec([
      (_: any) => _.Home.name("crm.Home").properties({ city: S.OfStr, zip: S.OfStr }),
      (_: any) => _.Knows.name("crm.Knows").links([(t: any) => t.name("knower"), (t: any) => t.name("known")])
        .properties({ since: S.OfInt }),
      (_: any) => _.Person.name("crm.Person").ref(true).properties({ name: S.OfStr, home: _.Home }).adjacencies([
        (t: any) => t.name("knows").relation([_.Knows]).me("knower"),
        (t: any) => t.relation([_.Knows]).me("known"),
        (t: any) => t.name("book").relation([_.Holds]).me("contact").owner(true),
        (t: any) => t.relation([_.Pinned]).me("pinner"), // no adjacency to crm.Pinned has a name
        (t: any) => t.relation([_.Pinned]).me("pinned"),
      ]),
      (_: any) => _.Pinned.name("crm.Pinned").links([(t: any) => t.name("pinner"), (t: any) => t.name("pinned")]),
      (_: any) => _.Holds.name("crm.Holds").links([(t: any) => t.name("book"), (t: any) => t.name("contact")])
        .properties({ key: S.OfStr }).uniques([["book", "key"], ["contact"]]),
      (_: any) => _.Book.name("crm.Book").ref(true).singleton("crm.TheBook").adjacencies({
        contacts: (t: any) => t.relation([_.Holds]).me("book") }),
      (_: any) => _.Note.name("crm.Note").ref(true).properties({ text: S.OfStr }).adjacencies({
        about: (t: any) => t.relation([_.About]).me("note") }),
      (_: any) => _.About.name("crm.About").links([(t: any) => t.name("note"), (t: any) => t.name("person")]),
    ])]))]);
}

const store: any = new B.Store();
store.transaction(crm);
const Person = store.registered("crm.Person");
const Book = store.registered("crm.Book");
const book = store.singleton("crm.TheBook");

/** A path, short: its singleton, then each key it passes, `crm.TheBook/ada`; a schema's, its name. */
function short(path: any): string {
  const parts: string[] = [path.singleton];
  for (let at = path.next ?? {}; at.entry; at = at.entry.next ?? {}) {
    parts.push(...Object.values(at.entry.properties ?? {}).map(String));
  }
  return parts[0] === "." ? parts.slice(1).join(".") : parts.join("/");
}

/** An effect, short: each symbol, as what it is bound to, then each mutation of its spec, in order. */
function shown(effect: any, source: any = store): string[] {
  const [pattern] = JSON.parse(Json.dumps(Plain.ToPlain(source).Transaction(effect))).mutations[0].upsert.exists;
  const lines: string[] = [];
  for (const [name, bound] of Object.entries(pattern.symbols) as [string, any][]) {
    lines.push("reference" in bound ? `${name} = ${short(bound.reference.path)}` : `${name} = new ${short(bound.schema[0].$ref)}`);
  }
  for (const spec of pattern.spec) {
    for (const [name, mutations] of Object.entries(spec) as [string, any[]][]) {
      for (const m of mutations) {
        if ("as_property" in m) {
          const [verb] = m.as_property.mutations;
          lines.push(`${name}.${m.as_property.name} = ` + ("set" in verb ? JSON.stringify(verb.set.value) : "cleared"));
          continue;
        }
        for (const v of m.as_adjacency.mutations) {
          const [[kind, body]] = Object.entries(v) as [string, any][];
          for (const entry of body.entries) {
            const parts = entry.mutations.map((x: any) => "as_link" in x
              ? `${x.as_link.name}=${x.as_link.mutations[0].set.ref.symbol}`
              : `${x.as_property.name}=${JSON.stringify(x.as_property.mutations[0].set.value)}`);
            lines.push(`${kind === "insert" ? "+" : "-"} ${name}.${m.as_adjacency.name}(${parts.join(", ")})`);
          }
        }
      }
    }
  }
  return lines;
}

/** Whether two plain values are the same, their keys in one order. */
function same(a: unknown, b: unknown): boolean {
  return JSON.stringify(a) === JSON.stringify(b);
}

/** A transaction adding persons to the book, each keyed by its name. */
function people(...names: string[]): any {
  return (t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ book: Book, ...Object.fromEntries(names.map((n) => [n, Person])) })
    .spec([(_: any) => _.book.contacts(Object.fromEntries(names.map((n) => [n, (e: any) => e.contact(_[n])]))),
      ...names.map((n) => (_: any) => _[n].name(n))])]))]);
}

const log: any[] = [];
const end = Changes.observe(store, (effect) => log.push(effect));

<a id="CHG-01"></a>
## CHG-01 · Each transaction that commits having changed something is given to the observer as its effect, after its commit; one that changes nothing is not

In [ ]:
store.transaction(people("ada", "bob"));
assert(same(shown(log.at(-1)), ["o0 = crm.TheBook", "o1 = new crm.Person", "o2 = new crm.Person",
  '+ o0.contacts(contact=o1, key="ada")', '+ o0.contacts(contact=o2, key="bob")', 'o1.name = "ada"', 'o2.name = "bob"']));
assert(log.at(-1).as_schema() === store.registered("Mutations.OfTransaction")); // schema data, as any value is
const count1 = log.length;
store.transaction(people("ada")); // changes nothing: the spec holds
assert(log.length === count1);

<a id="CHG-02"></a>
## CHG-02 · A transaction that raises, its writes dropped, is observed by none

In [ ]:
const ada = book.contacts({ key: "ada" });
const count2 = log.length;
raises(Error, () => store.transaction(() => {
  store.crm.Person(ada, (p: any) => p.name("Zed"));
  throw new Error("refused");
}), "refused");
assert(log.length === count2 && ada.name.get() === "ada");

<a id="CHG-03"></a>
## CHG-03 · What a transaction undid within itself is no change: an entry inserted and removed, a value set and set back

In [ ]:
const bob = book.contacts({ key: "bob" });
const count3 = log.length;
store.transaction(() => {
  store.crm.Person(ada, (p: any) => p.knows((a: any) => a.insert((e: any) => e.known(bob).since(2001n))));
  store.crm.Person(ada, (p: any) => p.knows((a: any) => a.remove((e: any) => e.known(bob))));
  store.crm.Person(ada, (p: any) => p.name("Ada"));
  store.crm.Person(ada, (p: any) => p.name("ada"));
});
assert(log.length === count3);

<a id="CHG-04"></a>
## CHG-04 · Removing an owning entry deletes what it owned: the effect removes that entry by its owner's key, naming nothing deleted, and what goes with it follows

In [ ]:
store.transaction(() => store.crm.Person(ada, (p: any) => p.knows((a: any) => a.insert((e: any) => e.known(bob).since(2001n)))));
assert(same(shown(log.at(-1)), ["o0 = crm.TheBook/bob", "o1 = crm.TheBook/ada", "+ o1.knows(known=o0, since=2001)"]));
store.transaction(() => store.crm.Book(book, (b: any) => b.contacts((a: any) => a.remove((e: any) => e.key("bob")))));
assert(same(shown(log.at(-1)), ["o0 = crm.TheBook", '- o0.contacts(key="bob")']));

<a id="CHG-05"></a>
## CHG-05 · A property changed is set to its value, written whole by its schema, a value object's part included, or cleared

In [ ]:
store.transaction(() => store.crm.Person(ada, (p: any) => p.home((h: any) => h.city("Paris").zip("75001"))));
store.transaction(() => store.crm.Person(ada, (p: any) => p.home((h: any) => h.city("Lyon"))));
assert(same(shown(log.at(-1)), ["o0 = crm.TheBook/ada", 'o0.home = {"city":"Lyon","zip":"75001"}']));
store.transaction(() => store.crm.Person(ada, (p: any) => p.name(null)));
assert(same(shown(log.at(-1)), ["o0 = crm.TheBook/ada", "o0.name = cleared"]));

<a id="CHG-06"></a>
## CHG-06 · A counted object, immutable, is written by value: bound to its schema, its properties and entries written, found by them

In [ ]:
store.transaction(() => store.crm.Note((n: any) => n.text("hello").about((a: any) => a.insert((e: any) => e.person(ada)))));
assert(same(shown(log.at(-1)), ["o0 = crm.TheBook/ada", "o1 = new crm.Note", "+ o1.about(person=o0)", 'o1.text = "hello"']));

<a id="CHG-07"></a>
## CHG-07 · Each effect, written and read as plain data and applied in order to a store as the first was, makes it the same, applied twice the same again; once the observing ends, none is given

In [ ]:
const replica: any = new B.Store();
replica.transaction(crm);
for (const effect of log) {
  const plain = Json.loads(Json.dumps(Plain.ToPlain(store).Transaction(effect)));
  for (let k = 0; k < 2; k++) replica.transaction(Plain.FromPlain(replica).Transaction(plain));
}
const data = (s: any): string => Json.dumps(Plain.ToPlain(s).Snapshot()[1]!); // its data, its schemas aside
assert(data(replica) === data(store));
end();
end(); // ending it again changes nothing
const count7 = log.length;
store.transaction(people("eve"));
assert(log.length === count7);

<a id="CHG-08"></a>
## CHG-08 · An entry that no object it links names an adjacency to cannot be written in the effect: while observed, its transaction is refused, naming its relation

In [ ]:
const pinned = store.registered("crm.Pinned");
const stop = Changes.observe(store, () => {});
raises(ValueError, () => store.transaction(() => { // only the framework's own writes reach such a relation
  Proxies._add_entry(pinned, Proxies._make_entry(pinned, new Map([["pinner", ada], ["pinned", ada]]), new Map()));
}), "no object an entry of 'crm.Pinned' links has an adjacency to it by name");
stop();
assert(Proxies._linking(pinned, "pinner", ada).length === 0);

<a id="CHG-09"></a>
## CHG-09 · Objects of many schemas owned through one relation: deleting one is written by its owner's key alone, with what it owned and every entry linking it following, applied twice the same; an owner with no adjacency by name to the relation cannot write it

In [ ]:
/** org.Team and org.Member, each owned through org.Owns, by name, as is org.Badge, a member's; members are buddies;
 * org.Key is owned by the singleton vault through no adjacency by name. */
function org(txn: any): any {
  const S = txn.Schemas;
  const owned = (t: any, _: any): any => t.name("owner").relation([_.Owns]).me("owned").owner(true);
  return txn.uid("org/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Owns: S.OfRelation, Root: S.OfObject, Team: S.OfObject, Member: S.OfObject, Badge: S.OfObject,
      Buddy: S.OfRelation, Holds: S.OfRelation, Vault: S.OfObject, Key: S.OfObject })
    .spec([
      (_: any) => _.Owns.name("org.Owns").links([(t: any) => t.name("owner"), (t: any) => t.name("owned")])
        .properties({ name: S.OfStr }).uniques([["owner", "name"], ["owned"]]),
      (_: any) => _.Buddy.name("org.Buddy").links([(t: any) => t.name("a"), (t: any) => t.name("b")]),
      (_: any) => _.Holds.name("org.Holds").links([(t: any) => t.name("vault"), (t: any) => t.name("key")])
        .properties({ name: S.OfStr }).uniques([["vault", "name"], ["key"]]),
      (_: any) => _.Root.name("org.Root").ref(true).singleton("org.TheRoot").adjacencies({
        owns: (t: any) => t.relation([_.Owns]).me("owner") }),
      (_: any) => _.Team.name("org.Team").ref(true).adjacencies([
        (t: any) => owned(t, _), (t: any) => t.name("owns").relation([_.Owns]).me("owner")]),
      (_: any) => _.Member.name("org.Member").ref(true).adjacencies([
        (t: any) => owned(t, _), (t: any) => t.name("owns").relation([_.Owns]).me("owner"),
        (t: any) => t.name("buddies").relation([_.Buddy]).me("a"),
        (t: any) => t.relation([_.Buddy]).me("b")]),
      (_: any) => _.Badge.name("org.Badge").ref(true).properties({ code: S.OfStr }).adjacencies([
        (t: any) => owned(t, _)]),
      (_: any) => _.Vault.name("org.Vault").ref(true).singleton("org.TheVault").adjacencies([
        (t: any) => t.relation([_.Holds]).me("vault")]), // no adjacency by name
      (_: any) => _.Key.name("org.Key").ref(true).adjacencies([
        (t: any) => t.name("vault").relation([_.Holds]).me("key").owner(true)]),
    ])]))]);
}

const first: any = new B.Store();
const second: any = new B.Store();
for (const s of [first, second]) s.transaction(org);
const effects: any[] = [];
Changes.observe(first, (effect) => effects.push(effect));
const root = first.singleton("org.TheRoot");
let team: any;
let ann: any;
let ben: any;
first.transaction(() => {
  team = first.org.Team((t: any) => t.owner((a: any) => a.insert((e: any) => e.name("core").owner(root))));
  ann = first.org.Member((m: any) => m.owner((a: any) => a.insert((e: any) => e.name("ann").owner(team))));
  ben = first.org.Member((m: any) => m.owner((a: any) => a.insert((e: any) => e.name("ben").owner(team)))
    .buddies((a: any) => a.insert((e: any) => e.b(ann))));
  first.org.Badge((b: any) => b.owner((a: any) => a.insert((e: any) => e.name("pass").owner(ann))).code("A1"));
});
first.transaction(() => first.org.Member(ben, (m: any) => m.buddies((a: any) => a.remove((e: any) => e.b(ann)))));
assert(same(shown(effects.at(-1), first), ["o0 = org.TheRoot/core/ann", "o1 = org.TheRoot/core/ben", "- o1.buddies(b=o0)"]));
first.transaction(() => first.org.Member(ben, (m: any) => m.buddies((a: any) => a.insert((e: any) => e.b(ann)))));
first.transaction(() => first.org.Team(team, (t: any) => t.owns((a: any) => a.remove((e: any) => e.name("ann")))));
assert(same(shown(effects.at(-1), first), ["o0 = org.TheRoot/core", '- o0.owns(name="ann")']));
for (const effect of effects) {
  const plain = Json.loads(Json.dumps(Plain.ToPlain(first).Transaction(effect)));
  for (let k = 0; k < 2; k++) second.transaction(Plain.FromPlain(second).Transaction(plain));
}
assert(data(second) === data(first) && same([...team.owns].map((e: any) => e.name.get()), ["ben"])
  && [...ben.buddies].length === 0);
const vault = first.singleton("org.TheVault");
let key: any;
first.transaction(() => {
  key = first.org.Key((k: any) => k.vault((a: any) => a.insert((e: any) => e.name("k1").vault(vault))));
});
raises(ValueError, () => first.transaction(() => first.org.Key(key, (k: any) => k.vault((a: any) => a.clear()))),
  "no owner of an entry of 'org.Holds' has an adjacency to it by name");